# Agentic Resume Screener — High-End Edition

### What's inside
| Feature | Details |
|---|---|
| **Multi-agent pipeline** | 9 nodes orchestrated by LangGraph |
| **LLM semantic matching** | Groq LLaMA 3.1 — catches `ML` ≈ `Machine Learning` |
| **Human-in-the-loop** | LangGraph interrupt + checkpointing — pauses for human review |
| **Batch screening** | Screen N resumes, auto-rank by score |
| **ChromaDB candidate store** | Persist every screened candidate as a vector |
| **RAG retrieval** | Query past candidates similar to a new JD |

```
parse_resume ──[error?]──► error ──► END
             └──────────► parse_jd ──► semantic_match ──► skill_gap
                                                              │
                                              interview_gen ◄─┘
                                                    │
                                                decision
                                            [needs review?]
                                          /              \
                                   human_review     store_candidate ──► END
                                        │
                                  store_candidate ──► END
```

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph groq PyPDF2 chromadb

## 2. Imports & API Key

In [ ]:
import os, json, hashlib
from groq import Groq
from typing import TypedDict, Dict, Any, List
import PyPDF2

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("Loaded API key from Colab secrets.")
except Exception:
    import getpass
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API key: ")
    print("API key set.")

## 3. Shared State
All nodes read from and write to a single typed state object. LangGraph merges the dicts each node returns back into this state.

In [ ]:
class State(TypedDict):
    # ── inputs ──────────────────────────────────────────────────────────
    resume_text:        str
    job_description:    str
    # ── parsed data ─────────────────────────────────────────────────────
    candidate_profile:  Dict[str, Any]
    job_requirements:   Dict[str, Any]
    semantic_match:     Dict[str, Any]
    # ── analysis ────────────────────────────────────────────────────────
    skill_gaps:         List[str]
    gap_recommendations:List[str]
    interview_questions:List[str]
    # ── decision ────────────────────────────────────────────────────────
    match_score:        float
    confidence:         float
    recommendation:     str
    requires_human:     bool
    reasoning_summary:  str
    # ── meta ────────────────────────────────────────────────────────────
    parsing_error:      bool
    error_message:      str

## 4. Utilities

In [ ]:
_client = None

def get_client():
    global _client
    if _client is None:
        _client = Groq(api_key=os.environ["GROQ_API_KEY"])
    return _client

def call_llm(prompt, model="llama-3.1-8b-instant"):
    resp = get_client().chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=model, temperature=0.0, max_tokens=2048
    )
    return resp.choices[0].message.content

def parse_pdf(file_path):
    try:
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            return "\n".join(page.extract_text() or "" for page in reader.pages).strip()
    except Exception as e:
        return f"ERROR_PARSING_RESUME: {e}"

def extract_json(text):
    try:
        s, e = text.find("{"), text.rfind("}")
        if s != -1 and e != -1:
            return json.loads(text[s:e+1])
    except Exception:
        pass
    return {}

def empty_state(resume_text, jd_text):
    return {
        "resume_text": resume_text, "job_description": jd_text,
        "candidate_profile": {}, "job_requirements": {}, "semantic_match": {},
        "skill_gaps": [], "gap_recommendations": [], "interview_questions": [],
        "match_score": 0.0, "confidence": 0.0, "recommendation": "",
        "requires_human": False, "reasoning_summary": "",
        "parsing_error": False, "error_message": ""
    }

## 5. Agent Nodes
### Nodes 1 & 2 — Resume Parser + JD Parser

In [ ]:
# ── Node 1: Resume Parser ───────────────────────────────────────────────
def resume_parser_node(state: State) -> dict:
    text = state["resume_text"]
    if "ERROR_PARSING_RESUME" in text:
        return {"candidate_profile": {}, "parsing_error": True, "error_message": text}

    prompt = f"""Extract info from this resume. Return ONLY valid JSON.
{{
    "name": "full name", "email": "email or empty",
    "years_experience": 0, "education": "degree and field",
    "institute": "college name",
    "skills": ["skill1"], "tools": ["tool1"],
    "domain_experience": ["domain1"],
    "has_internship": true, "has_projects": true, "certifications": []
}}
Resume:
{text[:3000]}
"""
    parsed = extract_json(call_llm(prompt))
    if not parsed:
        parsed = {"name": "Unknown", "email": "", "years_experience": 0,
                  "education": "Unknown", "institute": "", "skills": [],
                  "tools": [], "domain_experience": [],
                  "has_internship": False, "has_projects": False, "certifications": []}
    return {"candidate_profile": parsed, "parsing_error": False, "error_message": ""}


# ── Node 2: JD Parser ───────────────────────────────────────────────────
def jd_parser_node(state: State) -> dict:
    jd = state["job_description"]
    prompt = f"""Extract job requirements. Return ONLY valid JSON.
{{
    "role": "job title",
    "required_skills": ["skill1"], "preferred_skills": ["skill2"],
    "required_experience_years": 0, "required_education": "degree",
    "domain": "industry", "strictness_level": "strict or flexible or vague"
}}
Job Description:
{jd[:3000]}
"""
    parsed = extract_json(call_llm(prompt))
    if not parsed:
        parsed = {"role": "Unknown", "required_skills": [], "preferred_skills": [],
                  "required_experience_years": 0, "required_education": "",
                  "domain": "", "strictness_level": "vague"}
    return {"job_requirements": parsed}

### Node 3 — Semantic Match
LLM reasons about skill similarity instead of exact string comparison. Catches abbreviations, synonyms, and related technologies.

In [ ]:
def semantic_match_node(state: State) -> dict:
    p = state["candidate_profile"]
    j = state["job_requirements"]

    prompt = f"""You are an expert technical recruiter.
Semantically match candidate skills vs job requirements.
Consider synonyms: ML=Machine Learning, JS=JavaScript, Postgres=PostgreSQL, etc.

Candidate Skills    : {p.get("skills", [])}
Candidate Tools     : {p.get("tools", [])}
Experience          : {p.get("years_experience", 0)} yrs
Education           : {p.get("education", "")} @ {p.get("institute", "")}
Domains             : {p.get("domain_experience", [])}
Has Internship      : {p.get("has_internship", False)}

Required Skills     : {j.get("required_skills", [])}
Preferred Skills    : {j.get("preferred_skills", [])}
Required Exp        : {j.get("required_experience_years", 0)} yrs
Role                : {j.get("role", "")}

Return ONLY valid JSON:
{{
    "matched_skills": ["matched skill 1"],
    "missing_required_skills": ["missing skill 1"],
    "missing_preferred_skills": ["preferred skill 1"],
    "partial_matches": [{{"candidate_skill": "ML", "required_skill": "Machine Learning", "confidence": 0.9}}],
    "skill_match_score": 0.0,
    "experience_match_score": 0.0,
    "education_match_score": 0.0,
    "overall_fit": "strong or moderate or weak or poor"
}}
"""
    parsed = extract_json(call_llm(prompt))

    if not parsed:
        cand  = set(s.lower() for s in p.get("skills",[]) + p.get("tools",[]))
        req   = set(s.lower() for s in j.get("required_skills",[]))
        score = len(cand & req) / max(1, len(req))
        parsed = {"matched_skills": list(cand & req),
                  "missing_required_skills": list(req - cand),
                  "missing_preferred_skills": [],
                  "partial_matches": [],
                  "skill_match_score": score,
                  "experience_match_score": 0.5,
                  "education_match_score": 0.5,
                  "overall_fit": "moderate" if score > 0.5 else "poor"}

    # Weighted composite: skills 55% | experience 30% | education 15%
    score = round(
        0.55 * float(parsed.get("skill_match_score", 0))
        + 0.30 * float(parsed.get("experience_match_score", 0))
        + 0.15 * float(parsed.get("education_match_score", 0)), 2)
    return {"semantic_match": parsed, "match_score": score}

### Nodes 4 & 5 — Skill Gap Analysis + Interview Question Generator

In [ ]:
# ── Node 4: Skill Gap Analysis ─────────────────────────────────────────
def skill_gap_node(state: State) -> dict:
    missing = state["semantic_match"].get("missing_required_skills", [])
    if not missing:
        return {"skill_gaps": [],
                "gap_recommendations": ["No critical skill gaps. Candidate looks strong."]}

    p, j = state["candidate_profile"], state["job_requirements"]
    prompt = f"""Missing skills for {j.get("role","role")}: {missing}
Candidate: {p.get("education","")} — {p.get("domain_experience",[])}
Return ONLY valid JSON:
{{"prioritized_gaps": ["gap1","gap2"], "recommendations": ["rec1","rec2"]}}
"""
    parsed = extract_json(call_llm(prompt))
    return {
        "skill_gaps": parsed.get("prioritized_gaps", missing),
        "gap_recommendations": parsed.get("recommendations",
                               [f"Build a project using {s}" for s in missing])
    }


# ── Node 5: Interview Question Generator ────────────────────────────────
def interview_gen_node(state: State) -> dict:
    p, j, m = state["candidate_profile"], state["job_requirements"], state["semantic_match"]
    prompt = f"""Generate 5 targeted technical interview questions for:
Role              : {j.get("role","")}
Matched Skills    : {m.get("matched_skills",[])}
Skill Gaps        : {state.get("skill_gaps",[])}
Background        : {p.get("education","")} — {p.get("domain_experience",[])}
Return ONLY valid JSON: {{"questions": ["Q1?","Q2?","Q3?","Q4?","Q5?"]}}
"""
    parsed = extract_json(call_llm(prompt))
    return {"interview_questions": parsed.get("questions", [
        "Walk me through your most technically complex project.",
        "How do you approach learning a new framework quickly?",
        "Describe a hard bug you debugged — what was your process?",
        "How do you ensure code is production-ready?",
        "What drew you to this specific role?"
    ])}

### Nodes 6–9 — Decision Engine, Error Handler, Human Review, Candidate Store

In [ ]:
# ── Node 6: Decision Engine ─────────────────────────────────────────────
def decision_node(state: State) -> dict:
    score      = state["match_score"]
    match      = state["semantic_match"]
    job        = state["job_requirements"]
    profile    = state["candidate_profile"]
    strictness = job.get("strictness_level", "flexible")
    fit        = match.get("overall_fit", "moderate")

    thresholds = {
        "strict":   {"proceed": 0.80, "review": 0.55},
        "flexible": {"proceed": 0.65, "review": 0.40},
        "vague":    {"proceed": 0.60, "review": 0.35},
    }
    t = thresholds.get(strictness, thresholds["flexible"])

    if score >= t["proceed"] and fit in ("strong", "moderate"):
        rec, needs_human = "Proceed to Interview", False
        conf = min(0.85 + (score - t["proceed"]) * 0.5, 0.98)
    elif score >= t["review"] or fit == "moderate":
        rec, needs_human = "Needs Manual Review", True
        conf = min(0.65 + score * 0.2, 0.98)
    else:
        rec, needs_human = "Reject", False
        conf = min(0.80 + (t["review"] - score) * 0.5, 0.98)

    reasoning = (
        f"Score: {score:.0%}. Matched: {match.get('matched_skills',[])}. "
        f"Missing: {match.get('missing_required_skills',[])}. "
        f"Fit: {fit}. Strictness: {strictness}. "
        f"Candidate: {profile.get('name','Unknown')} — Role: {job.get('role','Unknown')}."
    )
    return {"recommendation": rec, "requires_human": needs_human,
            "confidence": round(conf, 2), "reasoning_summary": reasoning}


# ── Node 7: Error Handler ────────────────────────────────────────────────
def error_node(state: State) -> dict:
    return {"recommendation": "Error — Cannot Process", "requires_human": True,
            "confidence": 0.0, "match_score": 0.0,
            "reasoning_summary": f"Failed: {state.get('error_message','Unknown')}",
            "skill_gaps": [], "gap_recommendations": [], "interview_questions": []}


# ── Node 8: Human Review (interrupted before execution) ─────────────────
def human_review_node(state: State) -> dict:
    # LangGraph pauses BEFORE this node when interrupt_before=["human_review"]
    # Use graph.update_state() to override recommendation, then resume.
    return {}


# ── Node 9: Store Candidate in ChromaDB ─────────────────────────────────
_chroma_client = None
_candidates_db = None

def get_candidates_db():
    global _chroma_client, _candidates_db
    if _candidates_db is None:
        import chromadb
        _chroma_client = chromadb.Client()
        _candidates_db = _chroma_client.get_or_create_collection("candidates")
    return _candidates_db

def store_candidate_node(state: State) -> dict:
    db      = get_candidates_db()
    profile = state["candidate_profile"]
    name    = profile.get("name", "Unknown")
    doc = (
        f"Name: {name}. "
        f"Skills: {', '.join(profile.get('skills',[]))}. "
        f"Tools: {', '.join(profile.get('tools',[]))}. "
        f"Education: {profile.get('education','')} from {profile.get('institute','')}. "
        f"Domains: {', '.join(profile.get('domain_experience',[]))}. "
        f"Recommendation: {state['recommendation']}. Score: {state['match_score']}"
    )
    uid = f"{name.replace(' ','_').lower()}_{hashlib.md5(doc.encode()).hexdigest()[:8]}"
    try:
        db.add(documents=[doc],
               metadatas=[{"name": name,
                           "score": str(state["match_score"]),
                           "recommendation": state["recommendation"]}],
               ids=[uid])
        print(f"Stored {name} in ChromaDB (id={uid})")
    except Exception:
        pass  # Ignore duplicate IDs on re-runs
    return {}

## 6. Build the LangGraph

Two compiled versions:
- **`graph`** — checkpointing + `interrupt_before=["human_review"]` for interactive single screening
- **`batch_graph`** — no interrupt, used for bulk processing

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def route_after_resume(state: State) -> str:
    return "error" if state.get("parsing_error") else "parse_jd"

def route_after_decision(state: State) -> str:
    return "human_review" if state["requires_human"] else "store_candidate"

def build_workflow():
    wf = StateGraph(State)
    wf.add_node("parse_resume",   resume_parser_node)
    wf.add_node("parse_jd",       jd_parser_node)
    wf.add_node("semantic_match", semantic_match_node)
    wf.add_node("skill_gap",      skill_gap_node)
    wf.add_node("interview_gen",  interview_gen_node)
    wf.add_node("decision",       decision_node)
    wf.add_node("human_review",   human_review_node)
    wf.add_node("store_candidate",store_candidate_node)
    wf.add_node("error",          error_node)

    wf.set_entry_point("parse_resume")

    wf.add_conditional_edges("parse_resume", route_after_resume,
                             {"error": "error", "parse_jd": "parse_jd"})
    wf.add_edge("parse_jd",        "semantic_match")
    wf.add_edge("semantic_match",   "skill_gap")
    wf.add_edge("skill_gap",        "interview_gen")
    wf.add_edge("interview_gen",    "decision")
    wf.add_conditional_edges("decision", route_after_decision,
                             {"human_review": "human_review",
                              "store_candidate": "store_candidate"})
    wf.add_edge("human_review",     "store_candidate")
    wf.add_edge("store_candidate",  END)
    wf.add_edge("error",            END)
    return wf

# Interactive graph — pauses for human review when requires_human=True
memory     = MemorySaver()
graph      = build_workflow().compile(checkpointer=memory,
                                      interrupt_before=["human_review"])

# Batch graph — no interrupt, runs fully automatically
batch_graph = build_workflow().compile()

print("Both graphs compiled successfully.")

### Graph Visualisation

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 7. Single Screening — Human-in-the-Loop

When the AI marks a candidate as `Needs Manual Review`, the graph **pauses** before the `human_review` node. You see the AI's reasoning, can override the decision, then resume. The final state is checkpointed in memory.

In [ ]:
# ── Upload resume + JD (Colab) ─────────────────────────────────────────
try:
    from google.colab import files
    print("Upload resume PDF:")
    resume_file = list(files.upload().keys())[0]
    print("Upload job description TXT:")
    jd_file     = list(files.upload().keys())[0]
    resume_text = parse_pdf(resume_file)
    jd_text     = open(jd_file).read()
except Exception:
    # Local Jupyter — paste text below
    resume_text = """Paste resume text here"""
    jd_text     = """Paste JD text here"""

In [ ]:
# ── Run graph (pauses if human review needed) ───────────────────────────
import uuid
thread_id = str(uuid.uuid4())           # Unique ID for this screening session
config    = {"configurable": {"thread_id": thread_id}}

state = graph.invoke(empty_state(resume_text, jd_text), config)

print(f"\nAI Recommendation : {state['recommendation']}")
print(f"Match Score        : {state['match_score']:.0%}")
print(f"Confidence         : {state['confidence']:.0%}")
print(f"Requires Human     : {state['requires_human']}")
print(f"\nReasoning:\n{state['reasoning_summary']}")

In [ ]:
# ── Human review step (only runs when requires_human=True) ─────────────
if state["requires_human"]:
    print("\n" + "="*55)
    print("  HUMAN REVIEW REQUIRED")
    print("="*55)
    print(f"  AI says      : {state['recommendation']}")
    print(f"  Match Score  : {state['match_score']:.0%}")
    print(f"  Skill Gaps   : {state['skill_gaps']}")
    print("="*55)
    choice = input("\nYour decision [approve / reject / keep AI decision]: ").strip().lower()

    if choice == "approve":
        graph.update_state(config, {"recommendation": "Proceed to Interview (Human Approved)",
                                    "requires_human": False})
    elif choice == "reject":
        graph.update_state(config, {"recommendation": "Reject (Human Decision)",
                                    "requires_human": False})

    # Resume graph from checkpoint — runs human_review → store_candidate → END
    state = graph.invoke(None, config)
    print(f"\nFinal decision: {state['recommendation']}")
else:
    print("\nNo human review needed — decision was automatic.")

In [ ]:
# ── Display full result ─────────────────────────────────────────────────
from IPython.display import Markdown, display

gaps  = "\n".join(f"- {g}" for g in state["skill_gaps"]) or "None"
recs  = "\n".join(f"- {r}" for r in state["gap_recommendations"]) or "None"
qs    = "\n".join(f"{i+1}. {q}" for i,q in enumerate(state["interview_questions"]))

display(Markdown(f"""
## Screening Result

| Field | Value |
|---|---|
| **Candidate** | {state["candidate_profile"].get("name","Unknown")} |
| **Role** | {state["job_requirements"].get("role","Unknown")} |
| **Match Score** | {state["match_score"]:.0%} |
| **Confidence** | {state["confidence"]:.0%} |
| **Decision** | {state["recommendation"]} |
| **Human Reviewed** | {state["requires_human"]} |

### Reasoning
{state["reasoning_summary"]}

### Skill Gaps
{gaps}

### How to Bridge the Gaps
{recs}

### Interview Questions
{qs}
"""))

## 8. Batch Screening + Ranked Leaderboard

Upload multiple resumes at once. The system screens all of them against the same JD and returns a ranked leaderboard sorted by match score. Uses `batch_graph` (no interrupt) for fully automatic processing.

In [ ]:
def batch_screen(resume_paths, jd_text):
    results = []
    for i, path in enumerate(resume_paths, 1):
        print(f"Screening {i}/{len(resume_paths)}: {path}")
        resume_text = parse_pdf(path)
        result = batch_graph.invoke(empty_state(resume_text, jd_text))
        results.append(result)
    results.sort(key=lambda r: r["match_score"], reverse=True)
    return results


def display_leaderboard(results):
    rows = []
    icons = {"🟢": lambda r: r["match_score"] >= 0.65,    # green
             "🟡": lambda r: 0.40 <= r["match_score"] < 0.65, # yellow
             "🔴": lambda r: r["match_score"] < 0.40}      # red

    for rank, r in enumerate(results, 1):
        name = r["candidate_profile"].get("name", "Unknown")
        icon = next((k for k, fn in icons.items() if fn(r)), "⚪")
        rows.append(
            f"| {rank} | {icon} {name} | {r['match_score']:.0%} "
            f"| {r['recommendation']} | {r['confidence']:.0%} "
            f"| {r['skill_gaps'][:2]} |"
        )

    table = (
        "| Rank | Candidate | Score | Decision | Confidence | Top Gaps |\n"
        "|---|---|---|---|---|---|\n"
        + "\n".join(rows)
    )
    display(Markdown(f"## Candidate Leaderboard\n\n{table}"))

In [ ]:
# ── Upload multiple resumes (Colab) ─────────────────────────────────────
try:
    from google.colab import files
    print("Upload multiple resume PDFs (select all at once):")
    uploaded     = files.upload()
    resume_paths = list(uploaded.keys())

    # Re-use jd_text from the single screening cell above
    # Or define it here:
    # jd_text = """paste job description"""

except Exception:
    # Local Jupyter
    resume_paths = ["resume1.pdf", "resume2.pdf"]  # Update paths

batch_results = batch_screen(resume_paths, jd_text)
display_leaderboard(batch_results)

## 9. ChromaDB — Candidate Intelligence (RAG)

Every screened candidate is automatically stored in ChromaDB by `store_candidate_node`. The RAG query lets you ask: *"Who from my past candidates best fits this new JD?"* — without re-screening everyone.

In [ ]:
# ── Check how many candidates are stored ────────────────────────────────
db = get_candidates_db()
print(f"Candidates stored in ChromaDB: {db.count()}")

In [ ]:
# ── RAG: find past candidates similar to a new JD ───────────────────────
def find_similar_candidates(jd_query, n=3):
    db = get_candidates_db()
    if db.count() == 0:
        print("No candidates stored yet. Run screening first.")
        return

    results = db.query(
        query_texts=[jd_query[:500]],
        n_results=min(n, db.count())
    )

    rows = []
    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), 1
    ):
        similarity = round(1 - dist, 3)
        rows.append(
            f"| {i} | {meta['name']} | {float(meta['score']):.0%} "
            f"| {meta['recommendation']} | {similarity} |"
        )

    table = (
        "| Rank | Candidate | Match Score | Decision | Similarity |\n"
        "|---|---|---|---|---|\n"
        + "\n".join(rows)
    )
    display(Markdown(f"## Similar Past Candidates\n\n{table}"))


# ── Query example ───────────────────────────────────────────────────────
# Paste any new JD here — finds who from past screenings fits it best
new_jd = jd_text  # or paste a different JD
find_similar_candidates(new_jd, n=3)